In [ ]:
import os
from tqdm import tqdm
from dotenv import load_dotenv
from utils import get_stock_code_list, fetch_baostock_data, DATA_DIR
import clickhouse_connect
import pandas as pd
import numpy as np
import gc
import json
import atexit
import datetime

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS quant

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS quant.stock_daily_market
(
    `trade_date`   Date      COMMENT '交易日',
    `stock_code`   String    COMMENT '不带市场标识的股票代码',
    `open`         Float64   COMMENT '开盘价',
    `close`        Float64   COMMENT '收盘价',
    `high`         Float64   COMMENT '最高价',
    `low`          Float64   COMMENT '最低价',
    `volume`       Int64     COMMENT '成交量（单位：手）',
    `amount`       Float64   COMMENT '成交额（单位：元）',
    `pct_chg`      Float64   COMMENT '涨跌幅（单位：%）',
    `turnover_rate` Float64  COMMENT '换手率（单位：%）'
)
ENGINE = ReplacingMergeTree
PARTITION BY toYYYYMM(trade_date)
ORDER BY (stock_code, trade_date)
SETTINGS index_granularity = 8192;

In [ ]:
load_dotenv()
host = os.getenv('CH_HOST')
password = os.getenv('CH_PASSWORD')

client = clickhouse_connect.get_client(
    host=host,
    port=8123,
    username='default',
    password=password,
    database='quant',
    compress=True,
    connect_timeout=600,
    send_receive_timeout=6000
)

In [ ]:
log_events = []
LOG_FILE = DATA_DIR / f"stock_daily_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

start_date, end_date = "2018-01-01", "2026-01-01"

def add_log_event(level, message, details=None):
    event = {
        "timestamp": datetime.datetime.now().isoformat(),
        "level": level,
        "message": message
    }
    if details is not None:
        event["details"] = str(details)
    log_events.append(event)

def save_log():
    try:
        with open(LOG_FILE, 'w', encoding='utf-8') as f:
            json.dump(log_events, f, ensure_ascii=False, indent=2)
    except Exception as e:
        tqdm.write(f"CRITICAL: Failed to save log file: {e}")

atexit.register(save_log)

CHECKPOINT_FILE = DATA_DIR / f"stock_daily_checkpoint_{start_date}_{end_date}.json"

def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            return set(json.load(f))
    return set()

def save_checkpoint(processed_codes):
    try:
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump(list(processed_codes), f, ensure_ascii=False, indent=2)
    except Exception as e:
        add_log_event("WARNING", f"Failed to save checkpoint: {e}")

stock_code_lst = get_stock_code_list(filter_exchange=True)
processed_set = load_checkpoint()
codes_to_process = [code for code in stock_code_lst if code not in processed_set]
add_log_event("INFO", f"Total stocks: {len(stock_code_lst)}, already done: {len(processed_set)}, to process: {len(codes_to_process)}")

if not codes_to_process:
    add_log_event("INFO", "All stocks have been processed in previous runs. Exiting.")
    save_log()
    exit(0)


column_mapping = {
    'date': 'trade_date',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'Amount': 'amount',
    'pctChg': 'pct_chg',
    'turn': 'turnover_rate'
}
target_columns = [
    'trade_date', 'stock_code', 'open', 'close', 'high', 'low',
    'volume', 'amount', 'pct_chg', 'turnover_rate'
]
batch_size = 50
batch_dfs = []
failed_records = []

for code in tqdm(codes_to_process, desc="Fetching daily data"):
    try:
        raw_df = fetch_baostock_data(code, start_date, end_date)
        if raw_df is None or raw_df.empty:
            add_log_event("INFO", f"No data for {code}, skip")
            continue
        raw_df.reset_index(inplace=True)
        raw_df.rename(columns=column_mapping, inplace=True)
        raw_df['stock_code'] = code
        existing_cols = [col for col in target_columns if col in raw_df.columns]
        raw_df = raw_df[existing_cols]
        batch_dfs.append(raw_df)
    except Exception as e:
        add_log_event("ERROR", f"Error processing {code}: {e}", details=str(e))
        failed_records.append((code, "fetch/clean", str(e)))
        continue

    if len(batch_dfs) >= batch_size:
        try:
            full_df = pd.concat(batch_dfs, ignore_index=True)
            for col in target_columns:
                if col not in full_df.columns:
                    if col == 'volume':
                        full_df[col] = pd.NA
                    elif col in ('amount', 'pct_chg', 'turnover_rate'):
                        full_df[col] = np.nan
                    else:
                        full_df[col] = None
            full_df['trade_date'] = pd.to_datetime(full_df['trade_date'])
            full_df['volume'] = full_df['volume'].astype('Int64')
            full_df = full_df[target_columns]
            client.insert_df('stock_daily_market', full_df)

            # 插入成功：更新断点
            for df in batch_dfs:
                processed_set.add(df['stock_code'].iloc[0])
            save_checkpoint(processed_set)

        except Exception as e:
            batch_codes = [df['stock_code'].iloc[0] for df in batch_dfs]
            add_log_event("ERROR", f"Insert batch failed for {len(batch_codes)} stocks: {batch_codes}", details=str(e))
            for code in batch_codes:
                failed_records.append((code, "insert", str(e)))
        finally:
            batch_dfs.clear()
            if 'full_df' in locals():
                del full_df
            gc.collect()

if batch_dfs:
    try:
        full_df = pd.concat(batch_dfs, ignore_index=True)
        for col in target_columns:
            if col not in full_df.columns:
                if col == 'volume':
                    full_df[col] = pd.NA
                elif col in ('amount', 'pct_chg', 'turnover_rate'):
                    full_df[col] = np.nan
                else:
                    full_df[col] = None
        full_df['trade_date'] = pd.to_datetime(full_df['trade_date'])
        full_df['volume'] = full_df['volume'].astype('Int64')
        full_df = full_df[target_columns]
        client.insert_df('stock_daily_market', full_df)

        for df in batch_dfs:
            processed_set.add(df['stock_code'].iloc[0])
        save_checkpoint(processed_set)

    except Exception as e:
        batch_codes = [df['stock_code'].iloc[0] for df in batch_dfs]
        add_log_event("ERROR", f"Final insert failed for {len(batch_codes)} stocks: {batch_codes}", details=str(e))
        for code in batch_codes:
            failed_records.append((code, "insert", str(e)))
    finally:
        batch_dfs.clear()
        del full_df
        gc.collect()

if failed_records:
    add_log_event("ERROR", f"Total failures: {len(failed_records)} stocks")
    for code, stage, err in failed_records:
        add_log_event("ERROR", f"{code} ({stage}): {err}")
else:
    add_log_event("SUCCESS", "All stocks processed successfully.")

save_log()

In [ ]:
%%sql
SELECT count(*) FROM quant.stock_daily_market